# Nested Configs, Modularity, and Inheritance

## Overview

As your ML experiments grow in complexity, flat configurations become unwieldy. SpaX supports **modular configuration design** through three powerful patterns:

1. **Nested Configs**: Compose configs from smaller, reusable pieces
2. **Inheritance**: Extend and override base configurations
3. **Polymorphic Fields**: Use different config types for the same field

**What you'll learn:**
- Defining configs with Config-typed fields
- Subclassing configs to extend/override parameters
- Using Union types for polymorphic configuration fields
- Deep nesting (configs containing configs containing configs)
- Conditional logic on deeply nested fields
- When to use each pattern in practice

**Prerequisites:**
- Basic understanding of SpaX `Config` (see notebook 00)
- Familiarity with conditional parameters (helpful, see notebook 01)

**Why modular configs matter:**
- 🧩 **Reusability**: Define once, use everywhere
- 🎯 **Separation of concerns**: Model, training, data configs stay independent
- 🔧 **Maintainability**: Change one component without affecting others
- 🚀 **Flexibility**: Mix and match components for different experiments

Let's start with the simplest pattern: nested configs.

In [1]:
# Import SpaX
import spax as sp


# The simplest nested config: one config as a field of another
class OptimizerConfig(sp.Config):
    """Configuration for optimizer settings."""

    name: str = sp.Categorical(["adam", "sgd", "rmsprop"])
    learning_rate: float = sp.Float(ge=1e-5, le=1e-1, distribution="log")
    weight_decay: float = sp.Float(ge=0.0, le=0.1)


class TrainingConfig(sp.Config):
    """Training configuration with nested optimizer config."""

    # This field is another Config!
    optimizer: OptimizerConfig

    batch_size: int = sp.Int(ge=16, le=512)
    num_epochs: int = sp.Int(ge=1, le=100)


# Create an instance manually
config = TrainingConfig(
    optimizer=OptimizerConfig(name="adam", learning_rate=0.001, weight_decay=0.01),
    batch_size=32,
    num_epochs=10,
)

print("📦 Manually created nested config:")
print(config)
print()

# Random sampling works across nested configs!
print("🎲 Random sampling (seed=42):")
random_config = TrainingConfig.random(seed=42)
print(random_config)
print()

# Access nested fields with dot notation
print("🔍 Accessing nested fields:")
print(f"  Optimizer name: {random_config.optimizer.name}")
print(f"  Learning rate: {random_config.optimizer.learning_rate:.6f}")
print(f"  Batch size: {random_config.batch_size}")

📦 Manually created nested config:
TrainingConfig(optimizer=OptimizerConfig(name='adam', learning_rate=0.001, weight_decay=0.01), batch_size=32, num_epochs=10)

🎲 Random sampling (seed=42):
TrainingConfig(optimizer=OptimizerConfig(name='sgd', learning_rate=1.2590501261927484e-05, weight_decay=0.027502931836911926), batch_size=130, num_epochs=18)

🔍 Accessing nested fields:
  Optimizer name: sgd
  Learning rate: 0.000013
  Batch size: 130


## What Just Happened?

When you define a field with a `Config` type annotation, SpaX automatically:
- ✅ Treats it as a nested configuration
- ✅ Includes its parameters in random sampling
- ✅ Validates the nested structure
- ✅ Maintains the hierarchy in serialization

**Key observations:**
- `optimizer: OptimizerConfig` creates a nested config field
- Sampling automatically samples both parent and nested configs
- Access nested fields with dot notation: `config.optimizer.learning_rate`

**Visualizing the structure:**

Let's see how SpaX represents this hierarchy:

In [2]:
# Tree visualization shows the nested structure clearly
print("🌳 Tree view of nested configuration:\n")
print(TrainingConfig.get_tree())
print()

# Parameter names are hierarchical
print("📋 All searchable parameters:")
params = TrainingConfig.get_parameter_names()
for param in params:
    print(f"  • {param}")

🌳 Tree view of nested configuration:

TrainingConfig
├─ optimizer: OptimizerConfig
│  ├─ name: Categorical
│  │  ├─ 'adam'
│  │  ├─ 'sgd'
│  │  └─ 'rmsprop'
│  ├─ learning_rate: Float([1e-05, 0.1], log)
│  └─ weight_decay: Float([0.0, 0.1], uniform)
├─ batch_size: Int([16, 512], uniform)
└─ num_epochs: Int([1, 100], uniform)

📋 All searchable parameters:
  • TrainingConfig.optimizer::OptimizerConfig.name
  • TrainingConfig.optimizer::OptimizerConfig.learning_rate
  • TrainingConfig.optimizer::OptimizerConfig.weight_decay
  • TrainingConfig.batch_size
  • TrainingConfig.num_epochs


In [3]:
# Multiple nested configs: separate concerns cleanly
class ModelConfig(sp.Config):
    """Model architecture configuration."""

    num_layers: int = sp.Int(ge=1, le=12)
    hidden_dim: int = sp.Int(ge=64, le=1024)
    dropout: float = sp.Float(ge=0.0, le=0.5)


class DataConfig(sp.Config):
    """Data processing configuration."""

    train_split: float = sp.Float(ge=0.5, le=0.9)
    augmentation: bool = sp.Categorical([True, False])
    num_workers: int = sp.Int(ge=1, le=16)


class ExperimentConfig(sp.Config):
    """Complete experiment configuration with multiple nested configs."""

    model: ModelConfig
    training: TrainingConfig  # Reusing TrainingConfig from earlier!
    data: DataConfig

    # Top-level parameters
    experiment_name: str = "my_experiment"
    seed: int = sp.Int(ge=0, le=9999)


# Sample a complete experiment configuration
exp_config = ExperimentConfig.random(seed=123)

print("🧪 Complete experiment configuration:\n")
print(exp_config)
print("\n" + "=" * 60 + "\n")

# Access different nested configs
print("📊 Accessing different nested configs:")
print(f"  Model layers: {exp_config.model.num_layers}")
print(f"  Training optimizer: {exp_config.training.optimizer.name}")
print(f"  Training LR: {exp_config.training.optimizer.learning_rate:.6f}")
print(f"  Data augmentation: {exp_config.data.augmentation}")
print(f"  Experiment seed: {exp_config.seed}")

🧪 Complete experiment configuration:

ExperimentConfig(model=ModelConfig(num_layers=1, hidden_dim=338, dropout=0.04359333876131616), training=TrainingConfig(optimizer=OptimizerConfig(name='sgd', learning_rate=2.6965031984005817e-05, weight_decay=0.09011988779516947), batch_size=35, num_epochs=49), data=DataConfig(train_split=0.7144808160135707, augmentation=True, num_workers=2), experiment_name='my_experiment', seed=2615)


📊 Accessing different nested configs:
  Model layers: 1
  Training optimizer: sgd
  Training LR: 0.000027
  Data augmentation: True
  Experiment seed: 2615


## 🧩 Modularity Benefits

Notice how we **reused** `TrainingConfig` inside `ExperimentConfig`? This is the power of modular configs:

- **Define once, use everywhere**: `TrainingConfig` can be used in multiple experiment configs
- **Independent development**: Change `ModelConfig` without touching `TrainingConfig`
- **Clear organization**: Each config handles one concern (model, training, data)
- **Easy testing**: Test each config independently

---

## 🔄 Inheritance: Extending Configurations

Sometimes you want to create **variants** of a base configuration. SpaX supports standard Python inheritance:

**Use cases:**
- Create specialized versions of a base config
- Override default values or spaces
- Add new parameters to an existing config
- Build configuration hierarchies (e.g., `BaseModel` → `ResNet` → `ResNet50`)

Let's see it in action:

In [4]:
# Inheritance: Create specialized configs from base configs
class BaseModelConfig(sp.Config):
    """Base configuration for all models."""

    num_layers: int = sp.Int(ge=1, le=12)
    hidden_dim: int = sp.Int(ge=64, le=512)
    activation: str = sp.Categorical(["relu", "gelu"])


class ResNetConfig(BaseModelConfig):
    """ResNet-specific configuration - inherits from base."""

    # Add ResNet-specific parameters
    use_bottleneck: bool = sp.Categorical([True, False])
    stride: int = sp.Categorical([1, 2])

    # Override parent's hidden_dim with different range
    hidden_dim: int = sp.Int(ge=128, le=2048)  # ResNets typically use larger dims


class TransformerConfig(BaseModelConfig):
    """Transformer-specific configuration - inherits from base."""

    # Add Transformer-specific parameters
    num_heads: int = sp.Int(ge=1, le=16)
    feedforward_dim: int = sp.Int(ge=256, le=4096)

    # Override parent's activation choices
    activation: str = sp.Categorical(
        ["gelu", "swish"]
    )  # Transformers prefer gelu/swish


# Sample different model types
print("🎲 Sampling different model configurations:\n")

base_model = BaseModelConfig.random(seed=200)
print(
    f"Base Model:        layers={base_model.num_layers}, hidden={base_model.hidden_dim}, activation={base_model.activation}"
)

resnet = ResNetConfig.random(seed=201)
print(
    f"ResNet:            layers={resnet.num_layers}, hidden={resnet.hidden_dim}, activation={resnet.activation}"
)
print(f"                   bottleneck={resnet.use_bottleneck}, stride={resnet.stride}")

transformer = TransformerConfig.random(seed=202)
print(
    f"Transformer:       layers={transformer.num_layers}, hidden={transformer.hidden_dim}, activation={transformer.activation}"
)
print(
    f"                   heads={transformer.num_heads}, ff_dim={transformer.feedforward_dim}"
)

print("\n" + "=" * 60 + "\n")

# Tree views show the inheritance
print("🌳 ResNet tree (inherits + adds + overrides):\n")
print(ResNetConfig.get_tree())

🎲 Sampling different model configurations:

Base Model:        layers=1, hidden=168, activation=gelu
ResNet:            layers=2, hidden=1590, activation=relu
                   bottleneck=True, stride=1
Transformer:       layers=7, hidden=394, activation=gelu
                   heads=7, ff_dim=1058


🌳 ResNet tree (inherits + adds + overrides):

ResNetConfig
├─ num_layers: Int([1, 12], uniform)
├─ hidden_dim: Int([128, 2048], uniform)
├─ activation: Categorical
│  ├─ 'relu'
│  └─ 'gelu'
├─ use_bottleneck: Categorical
│  ├─ True
│  └─ False
└─ stride: Categorical
   ├─ 1
   └─ 2


## 🎭 Polymorphic Configs: Union Types

The most flexible pattern: a field that can be **one of several different Config types**.

**Syntax:**
```python
head: CNNConfig | TransformerConfig | LinearConfig
```

**Use cases:**
- Different model architectures (CNN vs Transformer encoder)
- Different optimizer types with unique parameters
- Pluggable components (different attention mechanisms, loss functions, etc.)
- Experiment with entirely different approaches in the same framework

**How it works:**
- SpaX treats Union of Configs as a `CategoricalSpace`
- Random sampling picks one config type and samples it
- Validation ensures the value matches one of the allowed types

Let's build a realistic example:

In [5]:
# Polymorphic configs: Different config types for the same field
class CNNEncoderConfig(sp.Config):
    """CNN-based encoder configuration."""

    num_conv_layers: int = sp.Int(ge=2, le=8)
    kernel_size: int = sp.Categorical([3, 5, 7])
    num_filters: int = sp.Int(ge=32, le=256)


class TransformerEncoderConfig(sp.Config):
    """Transformer-based encoder configuration."""

    num_layers: int = sp.Int(ge=2, le=12)
    num_heads: int = sp.Int(ge=4, le=16)
    hidden_dim: int = sp.Int(ge=128, le=1024)


class RNNEncoderConfig(sp.Config):
    """RNN-based encoder configuration."""

    num_layers: int = sp.Int(ge=1, le=6)
    hidden_size: int = sp.Int(ge=64, le=512)
    cell_type: str = sp.Categorical(["lstm", "gru"])


class FlexibleModelConfig(sp.Config):
    """Model with polymorphic encoder - can be CNN, Transformer, or RNN!"""

    # This field can be ANY of the three encoder types
    encoder: CNNEncoderConfig | TransformerEncoderConfig | RNNEncoderConfig

    output_dim: int = sp.Int(ge=10, le=1000)
    dropout: float = sp.Float(ge=0.0, le=0.5)


# Sample different configurations - each might have a different encoder type!
print("🎲 Sampling configurations with different encoder types:\n")

for i in range(5):
    config = FlexibleModelConfig.random(seed=300 + i)
    encoder_type = type(config.encoder).__name__
    print(f"Sample {i + 1}: encoder={encoder_type:25s} ", end="")

    if isinstance(config.encoder, CNNEncoderConfig):
        print(
            f"conv_layers={config.encoder.num_conv_layers}, filters={config.encoder.num_filters}"
        )
    elif isinstance(config.encoder, TransformerEncoderConfig):
        print(f"layers={config.encoder.num_layers}, heads={config.encoder.num_heads}")
    elif isinstance(config.encoder, RNNEncoderConfig):
        print(f"layers={config.encoder.num_layers}, cell={config.encoder.cell_type}")

print("\n" + "=" * 60 + "\n")
print("🌳 Tree view shows all possible encoder types:\n")
print(FlexibleModelConfig.get_tree())

🎲 Sampling configurations with different encoder types:

Sample 1: encoder=TransformerEncoderConfig  layers=7, heads=4
Sample 2: encoder=TransformerEncoderConfig  layers=9, heads=12
Sample 3: encoder=RNNEncoderConfig          layers=5, cell=gru
Sample 4: encoder=CNNEncoderConfig          conv_layers=7, filters=176
Sample 5: encoder=CNNEncoderConfig          conv_layers=5, filters=147


🌳 Tree view shows all possible encoder types:

FlexibleModelConfig
├─ encoder: Categorical
│  ├─ CNNEncoderConfig
│  │  ├─ num_conv_layers: Int([2, 8], uniform)
│  │  ├─ kernel_size: Categorical
│  │  │  ├─ 3
│  │  │  ├─ 5
│  │  │  └─ 7
│  │  └─ num_filters: Int([32, 256], uniform)
│  ├─ TransformerEncoderConfig
│  │  ├─ num_layers: Int([2, 12], uniform)
│  │  ├─ num_heads: Int([4, 16], uniform)
│  │  └─ hidden_dim: Int([128, 1024], uniform)
│  └─ RNNEncoderConfig
│     ├─ num_layers: Int([1, 6], uniform)
│     ├─ hidden_size: Int([64, 512], uniform)
│     └─ cell_type: Categorical
│        ├─ 'lstm'
│  

## 🏔️ Deep Nesting: Configs All The Way Down

Configs can be nested arbitrarily deep. This is useful for complex systems with multiple levels of hierarchy.

**Example structure:**
```
ExperimentConfig
  └─ model: ModelConfig
       └─ encoder: EncoderConfig
            └─ attention: AttentionConfig
                 └─ ... (and so on)
```

**When to use deep nesting:**
- ✅ Natural hierarchies (experiment → model → layer → sublayer)
- ✅ Clear separation at each level
- ⚠️ Don't overdo it - too deep becomes hard to navigate

Let's build a 3-level hierarchy and show how to work with it:

In [6]:
# Deep nesting: 3+ levels of hierarchy
class AttentionConfig(sp.Config):
    """Attention mechanism configuration (deepest level)."""

    num_heads: int = sp.Int(ge=1, le=16)
    head_dim: int = sp.Int(ge=32, le=128)
    use_flash_attention: bool = sp.Categorical([True, False])


class EncoderLayerConfig(sp.Config):
    """Single encoder layer configuration (middle level)."""

    attention: AttentionConfig
    feedforward_dim: int = sp.Int(ge=256, le=4096)
    dropout: float = sp.Float(ge=0.0, le=0.3)


class DeepModelConfig(sp.Config):
    """Model with deep nesting (top level)."""

    encoder: EncoderLayerConfig
    num_layers: int = sp.Int(ge=2, le=12)

    # Conditional based on DEEPLY nested field!
    # If attention has many heads, we need gradient checkpointing
    use_gradient_checkpointing: bool = sp.Conditional(
        sp.FieldCondition("encoder.attention.num_heads", sp.LargerThan(8)),
        true=True,
        false=False,
    )

    # Another deep condition: large head_dim needs more memory optimization
    mixed_precision: bool = sp.Conditional(
        sp.FieldCondition(
            "encoder.attention.head_dim", sp.LargerThan(64, or_equals=True)
        ),
        true=True,
        false=sp.Categorical([True, False]),  # Optional when head_dim is small
    )


# Sample and show deep access
print("🎲 Sampling deeply nested configs:\n")

for i in range(4):
    config = DeepModelConfig.random(seed=400 + i)

    # Access 3 levels deep!
    num_heads = config.encoder.attention.num_heads
    head_dim = config.encoder.attention.head_dim
    flash = config.encoder.attention.use_flash_attention

    print(f"Sample {i + 1}:")
    print(f"  encoder.attention.num_heads = {num_heads}")
    print(f"  encoder.attention.head_dim = {head_dim}")
    print(f"  encoder.attention.use_flash_attention = {flash}")
    print(
        f"  → use_gradient_checkpointing = {config.use_gradient_checkpointing} (auto: heads > 8)"
    )
    print(f"  → mixed_precision = {config.mixed_precision} (auto: head_dim >= 64)")
    print()

🎲 Sampling deeply nested configs:

Sample 1:
  encoder.attention.num_heads = 10
  encoder.attention.head_dim = 103
  encoder.attention.use_flash_attention = True
  → use_gradient_checkpointing = True (auto: heads > 8)
  → mixed_precision = True (auto: head_dim >= 64)

Sample 2:
  encoder.attention.num_heads = 16
  encoder.attention.head_dim = 44
  encoder.attention.use_flash_attention = True
  → use_gradient_checkpointing = True (auto: heads > 8)
  → mixed_precision = False (auto: head_dim >= 64)

Sample 3:
  encoder.attention.num_heads = 16
  encoder.attention.head_dim = 44
  encoder.attention.use_flash_attention = False
  → use_gradient_checkpointing = True (auto: heads > 8)
  → mixed_precision = False (auto: head_dim >= 64)

Sample 4:
  encoder.attention.num_heads = 8
  encoder.attention.head_dim = 111
  encoder.attention.use_flash_attention = True
  → use_gradient_checkpointing = False (auto: heads > 8)
  → mixed_precision = True (auto: head_dim >= 64)



In [7]:
# Tree view shows the deep hierarchy clearly
print("🌳 Tree view of deeply nested config:\n")
print(DeepModelConfig.get_tree())
print("\n" + "=" * 60 + "\n")


# Multi-field condition on deeply nested fields
class AdvancedDeepConfig(sp.Config):
    """Config with multi-field conditions on nested fields."""

    encoder: EncoderLayerConfig
    decoder: EncoderLayerConfig  # Reuse same structure for decoder

    num_layers: int = sp.Int(ge=2, le=12)

    # Complex condition: Use parameter sharing when BOTH encoder and decoder are large
    use_parameter_sharing: bool = sp.Conditional(
        sp.MultiFieldLambdaCondition(
            [
                "encoder.attention.num_heads",
                "decoder.attention.num_heads",
                "num_layers",
            ],
            lambda data: (
                data["encoder.attention.num_heads"] >= 8
                and data["decoder.attention.num_heads"] >= 8
                and data["num_layers"] > 6
            ),
        ),
        true=True,
        false=False,
    )


# Sample and test the multi-field condition
print("🧮 Multi-field condition on deeply nested fields:\n")

for i in range(4):
    config = AdvancedDeepConfig.random(seed=500 + i)

    enc_heads = config.encoder.attention.num_heads
    dec_heads = config.decoder.attention.num_heads
    layers = config.num_layers
    sharing = config.use_parameter_sharing

    print(
        f"Sample {i + 1}: enc_heads={enc_heads:2d}, dec_heads={dec_heads:2d}, layers={layers:2d}"
    )
    print(
        f"  → parameter_sharing={sharing} (enabled when: enc>=8 AND dec>=8 AND layers>6)"
    )
    print()

🌳 Tree view of deeply nested config:

DeepModelConfig
├─ encoder: EncoderLayerConfig
│  ├─ attention: AttentionConfig
│  │  ├─ num_heads: Int([1, 16], uniform)
│  │  ├─ head_dim: Int([32, 128], uniform)
│  │  └─ use_flash_attention: Categorical
│  │     ├─ True
│  │     └─ False
│  ├─ feedforward_dim: Int([256, 4096], uniform)
│  └─ dropout: Float([0.0, 0.3], uniform)
├─ mixed_precision: Conditional (if encoder.attention.head_dim >= 64)
│  ├─ true: True
│  └─ false: Categorical
│     ├─ True
│     └─ False
├─ num_layers: Int([2, 12], uniform)
└─ use_gradient_checkpointing: Conditional (if encoder.attention.num_heads > 8)
   ├─ true: True
   └─ false: False


🧮 Multi-field condition on deeply nested fields:

Sample 1: enc_heads=13, dec_heads=15, layers= 3
  → parameter_sharing=False (enabled when: enc>=8 AND dec>=8 AND layers>6)

Sample 2: enc_heads=11, dec_heads=10, layers= 9
  → parameter_sharing=True (enabled when: enc>=8 AND dec>=8 AND layers>6)

Sample 3: enc_heads= 6, dec_heads=12

## 🎯 Real-World Example: Putting It All Together

Let's build a realistic experiment configuration that combines:
- ✅ Multiple nested configs (separation of concerns)
- ✅ Inheritance (base → specialized models)
- ✅ Polymorphic fields (different model architectures)
- ✅ Deep nesting (3+ levels)
- ✅ Conditional logic on nested fields

This represents a typical production ML configuration:

In [8]:
# Comprehensive example: All patterns together
class BaseOptimizerConfig(sp.Config):
    """Base optimizer configuration."""

    learning_rate: float = sp.Float(ge=1e-5, le=1e-1, distribution="log")
    weight_decay: float = sp.Float(ge=0.0, le=0.1)


class AdamConfig(BaseOptimizerConfig):
    """Adam-specific optimizer."""

    beta1: float = sp.Float(ge=0.8, le=0.99)
    beta2: float = sp.Float(ge=0.9, le=0.999)


class SGDConfig(BaseOptimizerConfig):
    """SGD-specific optimizer."""

    momentum: float = sp.Float(ge=0.0, le=0.99)
    nesterov: bool = sp.Categorical([True, False])


class ProductionModelConfig(sp.Config):
    """Production model with all modular patterns."""

    # Polymorphic: Different architectures
    encoder: CNNEncoderConfig | TransformerEncoderConfig | RNNEncoderConfig

    # Polymorphic: Different optimizers
    optimizer: AdamConfig | SGDConfig

    # Nested structure
    num_layers: int = sp.Int(ge=2, le=12)
    output_dim: int = sp.Int(ge=10, le=1000)

    # Conditional on nested field type (using isinstance-style condition)
    use_layer_norm: bool = sp.Conditional(
        sp.FieldCondition("encoder", sp.IsInstance(TransformerEncoderConfig)),
        true=True,  # Always use layer norm for Transformers
        false=sp.Categorical([True, False]),  # Optional for others
    )

    # Conditional on deeply nested value (if it exists)
    gradient_clipping: float = sp.Conditional(
        sp.FieldCondition("optimizer", sp.IsInstance(AdamConfig)),
        true=sp.Float(ge=0.1, le=5.0),
        false=1.0,  # Fixed for SGD
    )


# Sample and display
print("🏭 Production configuration with all patterns:\n")

config = ProductionModelConfig.random(seed=600)
encoder_type = type(config.encoder).__name__
optimizer_type = type(config.optimizer).__name__

print(f"Encoder: {encoder_type}")
print(f"Optimizer: {optimizer_type}")
print(f"  LR: {config.optimizer.learning_rate:.6f}")
if isinstance(config.optimizer, AdamConfig):
    print(f"  Beta1: {config.optimizer.beta1:.3f}, Beta2: {config.optimizer.beta2:.3f}")
else:
    print(
        f"  Momentum: {config.optimizer.momentum:.3f}, Nesterov: {config.optimizer.nesterov}"
    )

print(f"Use layer norm: {config.use_layer_norm}")
print(f"Gradient clipping: {config.gradient_clipping:.2f}")

print("\n" + "=" * 60)
print("\n✨ All patterns working together seamlessly!")

🏭 Production configuration with all patterns:

Encoder: RNNEncoderConfig
Optimizer: SGDConfig
  LR: 0.000365
  Momentum: 0.904, Nesterov: False
Use layer norm: False
Gradient clipping: 1.00


✨ All patterns working together seamlessly!


## 📝 Summary: Building Modular Configurations

You've learned three powerful patterns for building complex, maintainable configurations:

### ✅ Pattern 1: Nested Configs
**What:** Config-typed fields create hierarchies
```python
class TrainingConfig(sp.Config):
    optimizer: OptimizerConfig  # Nested config
```
**When to use:**
- ✅ Separation of concerns (model, training, data)
- ✅ Reusable components
- ✅ Natural hierarchies in your system

### ✅ Pattern 2: Inheritance
**What:** Subclass configs to extend/override
```python
class ResNetConfig(BaseModelConfig):  # Inherits fields
    use_bottleneck: bool  # Add new fields
    hidden_dim: int = sp.Int(ge=128, le=2048)  # Override ranges
```
**When to use:**
- ✅ Creating specialized variants (ResNet50, ResNet101)
- ✅ Sharing common parameters across related configs
- ✅ Building config families (small/medium/large models)

### ✅ Pattern 3: Polymorphic Fields (Union Types)
**What:** A field that can be one of several Config types
```python
encoder: CNNConfig | TransformerConfig | RNNConfig
```
**When to use:**
- ✅ Experimenting with different architectures
- ✅ Pluggable components (different optimizers, heads, etc.)
- ✅ Maximum flexibility in configuration

### 🔗 Deep Nesting & Conditional Logic
**Access nested fields:**
```python
config.encoder.attention.num_heads  # Dot notation
```

**Conditions on nested fields:**
```python
sp.FieldCondition("encoder.attention.num_heads", sp.LargerThan(8))
```

**Multi-field conditions:**
```python
sp.MultiFieldLambdaCondition(
    ["encoder.attention.num_heads", "decoder.attention.num_heads"],
    lambda data: data["encoder.attention.num_heads"] >= 8 and ...
)
```

### 🎯 Key Takeaways
1. **Nest for organization** - Separate concerns, improve reusability
2. **Inherit for variants** - Create specialized versions efficiently
3. **Use Union for flexibility** - Support multiple architectures/approaches
4. **Go deep when natural** - Configs can nest arbitrarily deep
5. **Combine patterns** - Use all three together for maximum power

### 🚀 What's Next?
- **Notebook 03**: Serialization & persistence (JSON, YAML, TOML)
- **Notebook 04**: HPO with Optuna integration
- **Notebook 05**: Iterative refinement with overrides

**You now have the tools to build professional, modular ML configurations! 🎉**